In [ ]:
%load_ext autoreload
%autoreload 2
import hippo
import mrich
from mrich import print
from pathlib import Path
from os import environ
import shutil
import molparse as mp
from mocassin.mocassin import calculate_mocassin_tversky
import plotly.express as px

In [ ]:
target_name = "A71EV2A"
target_dir = Path(environ["BULK"]) / "TARGETS" / target_name
cycle_name = "cycle_01"
cycle_dir = Path(cycle_name)
aligned_dir = target_dir / "aligned_files"

In [ ]:
animal = hippo.HIPPO(target_name, target_dir / f"{target_name}.sqlite")

In [ ]:
pset = animal.poses(tag="fragmenstein_placed")

In [ ]:
scaffolds = animal.poses(tag="openbind_a71ev2a_c1_scaffolds")

In [ ]:
inspirations = {p.id:p for p in pset.inspirations}

In [ ]:
%%time
df = pset.get_df(
    alias=False, 
    smiles=False, 
    inchikey=False, 
    inspiration_ids=True, 
    mol=True, 
    distance_score=True, 
    energy_score=True
)

In [ ]:
df["inspiration_mols"] = df["inspiration_ids"].apply(lambda x: [inspirations[i].mol for i in x])

In [ ]:
ids = set(scaffolds.ids)
df["passed_bulkdock_filters"] = df.index.isin(ids)

In [ ]:
df

In [ ]:
df.to_pickle("openbind_a71ev2a_c1_placements_df.pkl.gz")

## Add Mocassin cols

In [ ]:
n = len(df)
for j,(i,row) in mrich.track(enumerate(df.iterrows()), total=n):

    mrich.set_progress_field("j", j)
    mrich.set_progress_field("n", n)
    
    try:
        combo, shape, colour = calculate_mocassin_tversky(
            row["inspiration_mols"],
            row["mol"],
            alpha=0.95,
            beta=0.05,
        )
        df.loc[i, "mocassin_combo(0.95,0.05)"] = combo
        df.loc[i, "mocassin_shape(0.95,0.05)"] = shape
        df.loc[i, "mocassin_colour(0.95,0.05)"] = colour
    except Exception as e:
        mrich.error(e)

In [ ]:
df.head()

In [ ]:
df.to_pickle("openbind_a71ev2a_c1_placements_df_scored.pkl.gz")

In [ ]:
fig = px.scatter(df, x="distance_score", y="mocassin_combo(0.95,0.05)", color="passed_bulkdock_filters")
fig

In [ ]:
fig.write_html("openbind_a71ev2a_c1_placements_mocassin.html")

In [ ]:
df["inspiration_names"] = df["inspiration_ids"].apply(lambda x: [inspirations[i].alias for i in x])

In [ ]:
fig = px.scatter(df, x="energy_score", y="mocassin_combo(0.95,0.05)", color="passed_bulkdock_filters")
fig

In [ ]:
%%time
tag_lookup = animal.db.get_pose_tag_dict()

In [ ]:
def get_method(i):
    tags = tag_lookup[i]
    return list(tags - {'fragmenstein_placed'})[0]

In [ ]:
df["method"] = df.index.to_series().apply(lambda x: get_method(x))

In [ ]:
df

In [ ]:
animal.P44391.draw()

In [ ]:
animal.P44391.compound.draw()

In [ ]:
animal.P44391.summary()

## Mocassin a/b/c plot

In [ ]:
n = len(df)
for j,(i,row) in mrich.track(enumerate(df.iterrows()), total=n):

    mrich.set_progress_field("j", j)
    mrich.set_progress_field("n", n)
    
    try:
        combo, shape, colour, a, b, c = calculate_mocassin_tversky(
            row["inspiration_mols"],
            row["mol"],
            alpha=0.95,
            beta=0.05,
        )
        df.loc[i, "mocassin_combo(0.95,0.05)"] = combo
        df.loc[i, "mocassin_shape(0.95,0.05)"] = shape
        df.loc[i, "mocassin_colour(0.95,0.05)"] = colour
        df.loc[i, "mocassin a"] = a
        df.loc[i, "mocassin b"] = b
        df.loc[i, "mocassin c"] = c
        df.loc[i, "mocassin a-c"] = a-c
        df.loc[i, "mocassin b-c"] = b-c
        df.loc[i, "mocassin a-c/a"] = (a-c)/a
        df.loc[i, "mocassin b-c/b"] = (b-c)/b
        # print(combo, shape, colour, a, b, c)
        # print(row)
    except Exception as e:
        mrich.error(e)

    # break

In [ ]:
fig = px.scatter(df, x="mocassin a-c/a", y="mocassin b-c/b", color="passed_bulkdock_filters", hover_data=["name"],)
fig

In [ ]:
fig.write_html("mocassin_tvserky_plot_2.html")

In [ ]:
animal.P45189.draw()

In [ ]:
animal.P53476.draw()

In [ ]:
animal.poses["C2091-P207"].draw()